# OV Circuit Analysis

Deep analysis of output value (OV) circuits: eigenvalue decomposition,
token copying strength, semantic role mapping, inter-layer composition,
and unembedding projection.

The OV circuit (W_V @ W_O) determines what each head writes to the residual stream.

## Why This Matters

OV circuit circuit analyzes the value-output projection that determines what information flows through attention. Understanding what each head copies when it attends to a position reveals the head's computational role in the model's circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.ov_circuit_analysis import (
    ov_eigenvalue_decomposition,
    token_copying_strength,
    ov_semantic_role,
    ov_composition_between_layers,
    ov_unembedding_projection,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## OV Eigenvalue Decomposition

In [ ]:
for l in range(min(2, cfg.n_layers)):
    for h in range(min(2, cfg.n_heads)):
        result = ov_eigenvalue_decomposition(model, layer=l, head=h)
        print(f"L{l}H{h}: trace={result['ov_trace']:.4f}, rank={result['ov_rank']:.1f}")
        print(f"  Top eigenvalues: {[f'{e:.3f}' for e in result['eigenvalues'][:3]]}")
        print(f"  Top SVs: {[f'{s:.3f}' for s in result['singular_values'][:3]]}")

## Token Copying Strength

In [ ]:
result = token_copying_strength(model, layer=0, head=0)
print(f"Mean copy strength: {result['mean_copy_strength']:.4f}")
print(f"Copy/suppress ratio: {result['copy_vs_suppress_ratio']:.4f}")
print(f"\nTop self-copiers:")
for tok, s in result['top_copied_tokens'][:5]:
    print(f"  Token {tok}: {s:.4f}")

## OV Semantic Role

In [ ]:
result = ov_semantic_role(model, tokens, layer=0, head=0)
print(f"Output norm: {result['output_norm']:.4f}")
print(f"Top promoted: {result['top_promoted_tokens']}")
print(f"Top demoted: {result['top_demoted_tokens']}")
print(f"Source contributions: {[f'{c:.3f}' for c in result['source_position_contributions']]}")

## OV Composition Between Layers

In [ ]:
for sl, sh in [(0, 0), (0, 1)]:
    for dl, dh in [(1, 0), (1, 1)]:
        result = ov_composition_between_layers(model, sl, sh, dl, dh)
        print(f"L{sl}H{sh}->L{dl}H{dh}: V={result['v_composition_score']:.3f}, "
              f"K={result['k_composition_score']:.3f}, Q={result['q_composition_score']:.3f}")

## OV Unembedding Projection

In [ ]:
result = ov_unembedding_projection(model, layer=0, head=0)
print(f"OV-logit rank: {result['ov_logit_matrix_rank']:.1f}")
print(f"Projection norm: {result['projection_norm']:.4f}")
print(f"\nTop positive: {result['top_positive_directions'][:5]}")
print(f"Top negative: {result['top_negative_directions'][:5]}")